# 🏥 Indian Pharmacy Inventory & Demand Forecasting
**Based on:** Pharma Sales Data — Milan Zdravković (Kaggle)

**Indianized:** ATC drug categories mapped to real Indian brand names, INR pricing, Indian seasonal demand patterns (monsoon fever spike, Diwali), and Indian pharmacy reorder logic.

### Indian Drug Mapping
| ATC Code | Generic | Indian Brands | Typical MRP (₹) |
|----------|---------|---------------|-----------------|
| M01AB | Diclofenac | Voveran, Voltaren, Diclomol | ₹25–₹80/strip |
| M01AE | Ibuprofen | Brufen, Combiflam, Ibugesic | ₹15–₹45/strip |
| N02BA | Aspirin | Disprin, Ecosprin, Delisprin | ₹10–₹30/strip |
| N02BE | Paracetamol | Calpol, Dolo 650, Crocin, Sumo | ₹10–₹25/strip |
| N05B  | Anxiolytics | Alprax, Restyl, Trika | ₹35–₹120/strip |
| N05C  | Sleeping aids | Nitrest, Nitravet, Stilnox | ₹40–₹150/strip |
| R03   | Respiratory | Asthalin, Seroflo, Foracort | ₹80–₹350/inhaler |
| R06   | Antihistamines | Allegra, Cetriz, Atarax, Zyrtec | ₹25–₹90/strip |

In [1]:
# Install if needed:
# pip install xgboost scikit-learn pandas numpy requests

import pandas as pd
import numpy as np
import warnings, json, pickle, os
from pathlib import Path
from io import StringIO

import requests
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

warnings.filterwarnings('ignore')
print('✅ Imports OK')

✅ Imports OK


In [2]:
# ══════════════════════════════════════════════
#  INDIAN PHARMACY CONFIGURATION
# ══════════════════════════════════════════════

INDIAN_DRUG_META = {
    'M01AB': {
        'generic':        'Diclofenac',
        'brands':         ['Voveran 50mg', 'Voltaren', 'Diclomol', 'Reactin'],
        'top_brand':      'Voveran 50mg',
        'manufacturer':   'Novartis India / Sun Pharma',
        'category':       'Anti-inflammatory (NSAID)',
        'form':           'Tablet / Gel',
        'mrp_per_strip':  45,       # INR
        'units_per_strip': 10,
        'lead_time_days':  3,
        'schedule':       'H',      # Indian drug schedule
        'gst_percent':    12,
        'seasonal_peaks': [1, 2, 6, 7, 8],   # Jan-Feb (winter), Jun-Aug (monsoon)
        'disease_season': 'Arthritis flares in winter, muscle pain year-round',
    },
    'M01AE': {
        'generic':        'Ibuprofen',
        'brands':         ['Brufen 400mg', 'Combiflam', 'Ibugesic Plus', 'Advil'],
        'top_brand':      'Brufen 400mg',
        'manufacturer':   'Abbott India / Pfizer',
        'category':       'Anti-inflammatory (NSAID)',
        'form':           'Tablet / Syrup',
        'mrp_per_strip':  30,
        'units_per_strip': 15,
        'lead_time_days':  3,
        'schedule':       'OTC',
        'gst_percent':    12,
        'seasonal_peaks': [6, 7, 8, 9],      # Monsoon — fever + cold surge
        'disease_season': 'High during monsoon fever season',
    },
    'N02BA': {
        'generic':        'Aspirin',
        'brands':         ['Disprin 350mg', 'Ecosprin 75mg', 'Delisprin', 'Loprin'],
        'top_brand':      'Ecosprin 75mg',
        'manufacturer':   'USV Ltd / Reckitt Benckiser',
        'category':       'Analgesic / Antiplatelet',
        'form':           'Tablet',
        'mrp_per_strip':  18,
        'units_per_strip': 14,
        'lead_time_days':  2,
        'schedule':       'OTC',
        'gst_percent':    12,
        'seasonal_peaks': [11, 12, 1, 2],    # Winter — cardiac events higher
        'disease_season': 'Used year-round for cardiac patients',
    },
    'N02BE': {
        'generic':        'Paracetamol',
        'brands':         ['Dolo 650', 'Calpol 500', 'Crocin 500', 'Sumo L', 'Febrex'],
        'top_brand':      'Dolo 650',
        'manufacturer':   'Micro Labs / GSK / Alkem',
        'category':       'Analgesic / Antipyretic',
        'form':           'Tablet / Syrup / Suspension',
        'mrp_per_strip':  30,
        'units_per_strip': 15,
        'lead_time_days':  1,
        'schedule':       'OTC',
        'gst_percent':    12,
        'seasonal_peaks': [6, 7, 8, 9, 10],  # HUGE monsoon surge (dengue, malaria, viral fever)
        'disease_season': 'Peak during monsoon (dengue/malaria fever), also winter flu',
    },
    'N05B': {
        'generic':        'Anxiolytics (Alprazolam / Diazepam)',
        'brands':         ['Alprax 0.25mg', 'Restyl 0.5mg', 'Trika 0.5mg', 'Lonazep'],
        'top_brand':      'Alprax 0.25mg',
        'manufacturer':   'Pfizer / Sun Pharma / Torrent',
        'category':       'Anxiolytic / Benzodiazepine',
        'form':           'Tablet',
        'mrp_per_strip':  55,
        'units_per_strip': 10,
        'lead_time_days':  4,
        'schedule':       'H / X',   # Psychotropic — requires special license
        'gst_percent':    12,
        'seasonal_peaks': [3, 4, 10, 11],   # Exam season (Mar-Apr), festival stress (Oct-Nov)
        'disease_season': 'Higher during exam season & festival periods',
    },
    'N05C': {
        'generic':        'Hypnotics / Sleeping aids (Nitrazepam / Zolpidem)',
        'brands':         ['Nitrest 5mg', 'Nitravet', 'Stilnox', 'Zolfresh'],
        'top_brand':      'Nitrest 5mg',
        'manufacturer':   'Sun Pharma / Sanofi',
        'category':       'Hypnotic / Sedative',
        'form':           'Tablet',
        'mrp_per_strip':  75,
        'units_per_strip': 10,
        'lead_time_days':  5,
        'schedule':       'H / X',
        'gst_percent':    12,
        'seasonal_peaks': [5, 6, 11, 12],
        'disease_season': 'Stress-related insomnia spikes in summer & year-end',
    },
    'R03': {
        'generic':        'Bronchodilators (Salbutamol / Budesonide)',
        'brands':         ['Asthalin 100mcg', 'Seroflo 250', 'Foracort 200', 'Budecort'],
        'top_brand':      'Asthalin Inhaler',
        'manufacturer':   'Cipla / Sun Pharma / Glenmark',
        'category':       'Respiratory / Bronchodilator',
        'form':           'Inhaler / Nebulisation solution',
        'mrp_per_strip':  180,
        'units_per_strip': 1,         # Inhalers — per unit
        'lead_time_days':  5,
        'schedule':       'H',
        'gst_percent':    5,
        'seasonal_peaks': [11, 12, 1, 2, 6, 7],  # Winter smog + monsoon mold triggers
        'disease_season': 'Winter smog (Delhi/NCR), monsoon mold spores',
    },
    'R06': {
        'generic':        'Antihistamines (Cetirizine / Fexofenadine)',
        'brands':         ['Cetriz 10mg', 'Allegra 120mg', 'Atarax 25mg', 'Histacet', 'Okacet'],
        'top_brand':      'Allegra 120mg',
        'manufacturer':   'Cipla / Sanofi / UCB',
        'category':       'Antihistamine',
        'form':           'Tablet / Syrup',
        'mrp_per_strip':  65,
        'units_per_strip': 10,
        'lead_time_days':  3,
        'schedule':       'OTC / H',
        'gst_percent':    12,
        'seasonal_peaks': [2, 3, 4, 7, 8],  # Spring pollen + monsoon allergy
        'disease_season': 'Spring allergies (Feb-Apr), monsoon dust mites (Jul-Aug)',
    },
}

# Indian seasonal multipliers — demand spikes by month
# Based on real Indian disease burden patterns
INDIAN_SEASONAL_MULTIPLIER = {
    1: 1.1,   # Jan: winter ailments
    2: 1.05,  # Feb: spring allergies starting
    3: 1.0,   # Mar: mild
    4: 1.05,  # Apr: summer heat, exam stress
    5: 1.1,   # May: hot weather illness
    6: 1.3,   # Jun: monsoon onset — big spike
    7: 1.5,   # Jul: PEAK monsoon — dengue/malaria/fever season
    8: 1.4,   # Aug: continued monsoon
    9: 1.2,   # Sep: post-monsoon viral
    10: 1.1,  # Oct: Navratri/Diwali (stress, overeating — digestive drugs up)
    11: 1.15, # Nov: winter approaching, pollution spike
    12: 1.2,  # Dec: winter illness
}

print('✅ Indian pharmacy configuration loaded')
print(f'   Drugs configured: {len(INDIAN_DRUG_META)}')
for code, meta in INDIAN_DRUG_META.items():
    print(f'   {code}: {meta["generic"]} → {meta["top_brand"]} (₹{meta["mrp_per_strip"]}/strip, Schedule {meta["schedule"]})')

✅ Indian pharmacy configuration loaded
   Drugs configured: 8
   M01AB: Diclofenac → Voveran 50mg (₹45/strip, Schedule H)
   M01AE: Ibuprofen → Brufen 400mg (₹30/strip, Schedule OTC)
   N02BA: Aspirin → Ecosprin 75mg (₹18/strip, Schedule OTC)
   N02BE: Paracetamol → Dolo 650 (₹30/strip, Schedule OTC)
   N05B: Anxiolytics (Alprazolam / Diazepam) → Alprax 0.25mg (₹55/strip, Schedule H / X)
   N05C: Hypnotics / Sleeping aids (Nitrazepam / Zolpidem) → Nitrest 5mg (₹75/strip, Schedule H / X)
   R03: Bronchodilators (Salbutamol / Budesonide) → Asthalin Inhaler (₹180/strip, Schedule H)
   R06: Antihistamines (Cetirizine / Fexofenadine) → Allegra 120mg (₹65/strip, Schedule OTC / H)


In [3]:
# ══════════════════════════════════════════════
#  LOAD & INDIANIZE THE KAGGLE DATASET
# ══════════════════════════════════════════════

# Option A: Download from GitHub mirror (no Kaggle login needed)
GITHUB_URL = (
    "https://raw.githubusercontent.com/mcallara/pharma-sales-data"
    "/main/salesdaily.csv"
)

def load_dataset():
    """Try GitHub mirror, fall back to local file."""
    try:
        print("Downloading dataset from GitHub...")
        resp = requests.get(GITHUB_URL, timeout=30)
        resp.raise_for_status()
        df = pd.read_csv(StringIO(resp.text))
        print(f"✅ Downloaded: {df.shape[0]} rows x {df.shape[1]} columns")
        return df
    except Exception as e:
        print(f"Download failed ({e})")
        print("Trying local file: Data/salesdaily.csv")
        try:
            df = pd.read_csv('Data/salesdaily.csv')
            print(f"✅ Local file loaded: {df.shape}")
            return df
        except FileNotFoundError:
            print("Generating synthetic Indian data...")
            return generate_indian_synthetic_data()

def generate_indian_synthetic_data():
    """
    Generate realistic synthetic Indian pharmacy sales data.
    Mirrors Kaggle dataset structure but with Indian demand patterns.
    """
    np.random.seed(42)
    dates = pd.date_range('2018-01-01', '2023-12-31', freq='D')
    drug_cols = list(INDIAN_DRUG_META.keys())

    # Base daily demand for Indian pharmacy (units/day)
    base_demand = {
        'M01AB': 18, 'M01AE': 22, 'N02BA': 15,
        'N02BE': 55, 'N05B': 8,   'N05C': 5,
        'R03':   12, 'R06':  20,
    }

    rows = []
    for date in dates:
        row = {'datum': date.strftime('%Y-%m-%d')}
        season_mult = INDIAN_SEASONAL_MULTIPLIER.get(date.month, 1.0)
        is_weekend  = 1 if date.dayofweek >= 5 else 0

        for drug in drug_cols:
            base   = base_demand[drug]
            # Drug-specific seasonal multiplier
            peaks  = INDIAN_DRUG_META[drug]['seasonal_peaks']
            drug_seasonal = 1.4 if date.month in peaks else 1.0
            # Weekend slight dip
            wknd   = 0.85 if is_weekend else 1.0
            # Random noise
            noise  = np.random.normal(1.0, 0.12)
            # Year-over-year growth (~6% annually — Indian pharma CAGR)
            yoy    = 1 + (date.year - 2018) * 0.06

            demand = max(0, round(base * season_mult * drug_seasonal * wknd * noise * yoy, 2))
            row[drug] = demand

        rows.append(row)

    df = pd.DataFrame(rows)
    print(f"✅ Synthetic Indian data generated: {df.shape}")
    return df

df_raw = load_dataset()
print('\nColumns:', df_raw.columns.tolist())
print(df_raw.head(3))

✅ Downloaded: 2106 rows x 13 columns

Columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']
      datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C   R03  R06  Year  Month  \
0  1/2/2014    0.0   3.67    3.4  32.40   7.0   0.0   0.0  2.0  2014      1   
1  1/3/2014    8.0   4.00    4.4  50.60  16.0   0.0  20.0  4.0  2014      1   
2  1/4/2014    2.0   1.00    6.5  61.85  10.0   0.0   9.0  1.0  2014      1   

   Hour Weekday Name  
0   248     Thursday  
1   276       Friday  
2   276     Saturday  


In [4]:
# ══════════════════════════════════════════════
#  PREPROCESSING & FEATURE ENGINEERING
# ══════════════════════════════════════════════

df_raw['datum'] = pd.to_datetime(df_raw['datum'])
df_raw.rename(columns={'datum': 'date'}, inplace=True)
df_raw.sort_values('date', inplace=True)
df_raw.reset_index(drop=True, inplace=True)

drug_cols = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']

# Melt wide → long format (one row per drug per day)
df_long = df_raw.melt(id_vars=['date'], value_vars=drug_cols,
                       var_name='drug_code', value_name='quantity_dispensed')

# Add Indian drug metadata
df_long['generic_name']   = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['generic'])
df_long['top_brand']      = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['top_brand'])
df_long['category']       = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['category'])
df_long['mrp_per_strip']  = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['mrp_per_strip'])
df_long['lead_time_days'] = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['lead_time_days'])
df_long['schedule']       = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['schedule'])
df_long['gst_pct']        = df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['gst_percent'])
df_long['units_per_strip']= df_long['drug_code'].map(lambda x: INDIAN_DRUG_META[x]['units_per_strip'])

# Revenue in INR
df_long['revenue_inr'] = df_long['quantity_dispensed'] * df_long['mrp_per_strip']

# Date features
df_long['day_of_week']       = df_long['date'].dt.dayofweek
df_long['day_of_month']      = df_long['date'].dt.day
df_long['month']             = df_long['date'].dt.month
df_long['year']              = df_long['date'].dt.year
df_long['week_of_year']      = df_long['date'].dt.isocalendar().week.astype(int)
df_long['quarter']           = df_long['date'].dt.quarter
df_long['is_weekend']        = (df_long['day_of_week'] >= 5).astype(int)
df_long['is_month_start']    = df_long['date'].dt.is_month_start.astype(int)
df_long['is_month_end']      = df_long['date'].dt.is_month_end.astype(int)
df_long['indian_season_mult']= df_long['month'].map(INDIAN_SEASONAL_MULTIPLIER)

# Indian festival flag (approx dates)
df_long['is_festival_week'] = (
    ((df_long['month'] == 10) & (df_long['day_of_month'].between(15, 31))) |  # Diwali
    ((df_long['month'] == 3)  & (df_long['day_of_month'].between(15, 31))) |  # Holi
    ((df_long['month'] == 8)  & (df_long['day_of_month'].between(10, 20)))    # Independence Day
).astype(int)

# Monsoon flag (June–September)
df_long['is_monsoon'] = df_long['month'].isin([6, 7, 8, 9]).astype(int)

# Lag features (per drug)
df_long.sort_values(['drug_code', 'date'], inplace=True)
for lag in [1, 7, 14, 30]:
    df_long[f'lag_{lag}'] = df_long.groupby('drug_code')['quantity_dispensed'].shift(lag)

# Rolling features
for window in [7, 14, 30, 90]:
    df_long[f'roll_mean_{window}'] = (
        df_long.groupby('drug_code')['quantity_dispensed']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df_long[f'roll_std_{window}'] = (
        df_long.groupby('drug_code')['quantity_dispensed']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )

# Trend feature
df_long['demand_trend'] = (
    df_long.groupby('drug_code')['quantity_dispensed']
    .transform(lambda x: x.shift(1).rolling(30, min_periods=7).mean() -
                         x.shift(1).rolling(90, min_periods=30).mean())
)

# Simulate stock levels
def simulate_stock(group):
    avg = group['quantity_dispensed'].mean()
    stock, stocks = avg * 30, []
    for qty in group['quantity_dispensed']:
        stocks.append(round(stock))
        stock = max(0, stock - qty)
        if stock < avg * 7:
            stock += avg * 30
    group = group.copy()
    group['stock_level'] = stocks
    return group

df_long = df_long.groupby('drug_code', group_keys=False).apply(simulate_stock)

# Stock-derived features
df_long['stock_lag1']     = df_long.groupby('drug_code')['stock_level'].shift(1)
df_long['days_of_stock']  = df_long['stock_level'] / (df_long['roll_mean_30'] + 0.01)

# Safety stock (Indian formula: avg_demand × lead_time × 1.5 safety factor)
df_long['safety_stock']   = df_long['roll_mean_30'] * df_long['lead_time_days'] * 1.5
df_long['reorder_point']  = df_long['roll_mean_30'] * df_long['lead_time_days'] + df_long['safety_stock']

# Stock status
def classify_stock(row):
    dos = row['days_of_stock']
    if dos < 3:   return 'CRITICAL'
    if dos < 7:   return 'LOW'
    if dos < 14:  return 'ADEQUATE'
    if dos < 30:  return 'GOOD'
    return 'OVERSTOCK'

df_long['stock_status'] = df_long.apply(classify_stock, axis=1)

# Encode categoricals
le_drug = LabelEncoder()
le_cat  = LabelEncoder()
df_long['drug_enc']     = le_drug.fit_transform(df_long['drug_code'])
df_long['category_enc'] = le_cat.fit_transform(df_long['category'])

df = df_long.dropna().reset_index(drop=True)
print(f'✅ Feature engineering done. Shape: {df.shape}')
print(f'   Date range: {df.date.min().date()} → {df.date.max().date()}')
print(f'\nStock status distribution:')
print(df['stock_status'].value_counts())

✅ Feature engineering done. Shape: (16608, 45)
   Date range: 2014-02-01 → 2019-10-08

Stock status distribution:
stock_status
GOOD         8109
OVERSTOCK    4630
ADEQUATE     3410
LOW           459
Name: count, dtype: int64


In [5]:
# ══════════════════════════════════════════════
#  MODEL A — DEMAND FORECASTING (XGBoost)
# ══════════════════════════════════════════════

FEATURES = [
    'drug_enc', 'category_enc',
    'day_of_week', 'day_of_month', 'month', 'year',
    'week_of_year', 'quarter', 'is_weekend',
    'is_month_start', 'is_month_end',
    'is_festival_week', 'is_monsoon', 'indian_season_mult',
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_30', 'roll_mean_90',
    'roll_std_7', 'roll_std_30',
    'demand_trend',
    'stock_level', 'stock_lag1',
    'lead_time_days', 'mrp_per_strip', 'gst_pct',
]

X = df[FEATURES].astype('float64')
y = df['quantity_dispensed'].astype('float64')

# Time-based split — NEVER shuffle for time series!
split = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

model_demand = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
)

model_demand.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100,
)

y_pred = model_demand.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)
mape   = np.mean(np.abs((y_test - y_pred) / (y_test + 0.01))) * 100

print(f'\n📊 Demand Forecast Results:')
print(f'   MAE  : {mae:.2f} units/day')
print(f'   RMSE : {rmse:.2f}')
print(f'   R²   : {r2:.4f}')
print(f'   MAPE : {mape:.1f}%')

Train: (13286, 30) | Test: (3322, 30)
[0]	validation_0-rmse:6.36182
[100]	validation_0-rmse:4.42406
[200]	validation_0-rmse:4.42736
[300]	validation_0-rmse:4.45079
[399]	validation_0-rmse:4.47616

📊 Demand Forecast Results:
   MAE  : 2.95 units/day
   RMSE : 4.48
   R²   : 0.1539
   MAPE : 6138.5%


In [6]:
# ══════════════════════════════════════════════
#  MODEL B — STOCK STATUS CLASSIFIER (XGBoost)
# ══════════════════════════════════════════════

le_status = LabelEncoder()
y_cls = le_status.fit_transform(df['stock_status'])
X_cls = df[FEATURES].astype('float64')

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=42, stratify=y_cls)

model_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
model_clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

y_pred_cls = model_clf.predict(X_te)
print('📊 Stock Status Classification Report:')
print(classification_report(y_te, y_pred_cls, target_names=le_status.classes_))

📊 Stock Status Classification Report:
              precision    recall  f1-score   support

    ADEQUATE       0.96      0.98      0.97       682
        GOOD       0.98      0.98      0.98      1622
         LOW       0.99      0.87      0.92        92
   OVERSTOCK       0.98      0.98      0.98       926

    accuracy                           0.98      3322
   macro avg       0.98      0.95      0.96      3322
weighted avg       0.98      0.98      0.98      3322



In [7]:
# ══════════════════════════════════════════════
#  INDIAN PHARMACY INVENTORY DASHBOARD
# ══════════════════════════════════════════════

def build_inventory_report(df, drug_meta):
    """Generate full Indian pharmacy inventory report with reorder recommendations."""
    last = df.sort_values('date').groupby('drug_code').last().reset_index()

    rows = []
    for _, row in last.iterrows():
        code  = row['drug_code']
        meta  = drug_meta[code]
        stock = row['stock_level']
        avg   = row['roll_mean_30'] if row['roll_mean_30'] > 0 else 1
        dos   = stock / avg
        rop   = row['reorder_point']
        ss    = row['safety_stock']

        # EOQ — Economic Order Quantity (Wilson formula)
        ordering_cost  = 500   # INR per order (Indian distributor)
        holding_pct    = 0.25  # 25% of MRP per year
        holding_cost   = meta['mrp_per_strip'] * holding_pct / 365
        annual_demand  = avg * 365
        eoq = np.sqrt((2 * annual_demand * ordering_cost) / (holding_cost + 0.01))

        # Status
        if dos < 3:   status = '🚨 CRITICAL'
        elif dos < 7: status = '⚠️ LOW'
        elif dos > 45: status = '📦 OVERSTOCK'
        else:         status = '✅ OK'

        # Action
        if stock <= rop:
            order_qty = max(eoq, avg * 14)   # at least 2 weeks
            action = f'ORDER {round(order_qty):.0f} units from {meta["manufacturer"].split("/")[0].strip()}'
        elif dos > 45:
            action = 'REDUCE next order — overstock risk'
        else:
            action = 'Monitor — no action needed'

        rows.append({
            'Drug Code':       code,
            'Generic':         meta['generic'],
            'Top Brand':       meta['top_brand'],
            'Schedule':        meta['schedule'],
            'Stock (units)':   round(stock),
            'Days of Stock':   round(dos, 1),
            'Avg Daily Demand':round(avg, 1),
            'Reorder Point':   round(rop),
            'Safety Stock':    round(ss),
            'EOQ (units)':     round(eoq),
            'MRP/Strip (₹)':   meta['mrp_per_strip'],
            'Stock Value (₹)': round(stock * meta['mrp_per_strip'] / meta['units_per_strip']),
            'Status':          status,
            'Action':          action,
            'Season Note':     meta['disease_season'],
        })

    return pd.DataFrame(rows)

report = build_inventory_report(df, INDIAN_DRUG_META)

print('=' * 80)
print('🏥  INDIAN PHARMACY INVENTORY REPORT')
print('=' * 80)
display_cols = ['Drug Code','Generic','Top Brand','Stock (units)','Days of Stock',
                'Status','Reorder Point','EOQ (units)','Action']
print(report[display_cols].to_string(index=False))

print('\n💰 STOCK VALUE SUMMARY')
total_val = report['Stock Value (₹)'].sum()
print(f'   Total Inventory Value : ₹{total_val:,.0f}')
print(f'   GST Applicable        : ~₹{total_val * 0.12:,.0f} (avg 12% GST)')

print('\n🚨 CRITICAL / LOW STOCK (needs immediate action):')
urgent = report[report['Status'].str.contains('CRITICAL|LOW', na=False)]
if len(urgent):
    print(urgent[['Drug Code','Generic','Top Brand','Stock (units)','Days of Stock','Action']].to_string(index=False))
else:
    print('   None — all drugs adequately stocked')

🏥  INDIAN PHARMACY INVENTORY REPORT
Drug Code                                           Generic        Top Brand  Stock (units)  Days of Stock Status  Reorder Point  EOQ (units)                     Action
    M01AB                                        Diclofenac     Voveran 50mg            121           21.4   ✅ OK             42         7106 Monitor — no action needed
    M01AE                                         Ibuprofen     Brufen 400mg             95           23.4   ✅ OK             30         6970 Monitor — no action needed
    N02BA                                           Aspirin    Ecosprin 75mg             94           29.4   ✅ OK             16         7229 Monitor — no action needed
    N02BE                                       Paracetamol         Dolo 650            762           21.0   ✅ OK             91        20832 Monitor — no action needed
     N05B               Anxiolytics (Alprazolam / Diazepam)    Alprax 0.25mg            232           28.8   ✅ OK      

In [8]:
# ══════════════════════════════════════════════
#  14-DAY DEMAND FORECAST — DRUG-WISE
# ══════════════════════════════════════════════

def forecast_next_14_days(drug_code, df, model, features):
    """Forecast next 14 days demand for a given drug."""
    drug_df   = df[df['drug_code'] == drug_code].copy().sort_values('date')
    last_row  = drug_df.iloc[-1].copy()
    last_date = drug_df['date'].iloc[-1]
    meta      = INDIAN_DRUG_META[drug_code]

    results = []
    for i in range(1, 15):
        fdate = last_date + pd.Timedelta(days=i)
        inp   = last_row[features].copy()
        inp['day_of_week']        = fdate.dayofweek
        inp['day_of_month']       = fdate.day
        inp['month']              = fdate.month
        inp['week_of_year']       = fdate.isocalendar()[1]
        inp['quarter']            = (fdate.month - 1) // 3 + 1
        inp['is_weekend']         = int(fdate.dayofweek >= 5)
        inp['is_month_start']     = int(fdate.day == 1)
        inp['is_month_end']       = int(fdate.day == fdate.days_in_month)
        inp['is_monsoon']         = int(fdate.month in [6, 7, 8, 9])
        inp['indian_season_mult'] = INDIAN_SEASONAL_MULTIPLIER.get(fdate.month, 1.0)
        inp['is_festival_week']   = int(
            (fdate.month == 10 and 15 <= fdate.day <= 31) or
            (fdate.month == 3  and 15 <= fdate.day <= 31) or
            (fdate.month == 8  and 10 <= fdate.day <= 20)
        )

        pred = max(0, model.predict(pd.DataFrame([inp]))[0])
        results.append({
            'Date':        fdate.date(),
            'Day':         fdate.strftime('%a'),
            'Predicted':   round(pred, 1),
            'Revenue_INR': round(pred * meta['mrp_per_strip'], 0),
        })

    return pd.DataFrame(results)

print('14-DAY DEMAND FORECAST — Indian Pharmacy')
print('=' * 55)

for code in drug_cols:
    meta = INDIAN_DRUG_META[code]
    fc = forecast_next_14_days(code, df, model_demand, FEATURES)
    total_14d = fc['Predicted'].sum()
    revenue_14d = fc['Revenue_INR'].sum()
    print(f'\n{code} — {meta["generic"]} ({meta["top_brand"]})')
    print(f'  14-day forecast: {total_14d:.0f} units | ₹{revenue_14d:,.0f} revenue')
    print(fc.to_string(index=False))

14-DAY DEMAND FORECAST — Indian Pharmacy

M01AB — Diclofenac (Voveran 50mg)
  14-day forecast: 75 units | ₹3,367 revenue
      Date Day  Predicted  Revenue_INR
2019-10-09 Wed        5.4        244.0
2019-10-10 Thu        6.0        272.0
2019-10-11 Fri        6.3        284.0
2019-10-12 Sat        6.5        293.0
2019-10-13 Sun        5.7        255.0
2019-10-14 Mon        4.9        219.0
2019-10-15 Tue        4.9        223.0
2019-10-16 Wed        4.8        217.0
2019-10-17 Thu        4.7        213.0
2019-10-18 Fri        5.0        226.0
2019-10-19 Sat        5.4        243.0
2019-10-20 Sun        5.5        249.0
2019-10-21 Mon        4.8        214.0
2019-10-22 Tue        4.8        215.0

M01AE — Ibuprofen (Brufen 400mg)
  14-day forecast: 49 units | ₹1,482 revenue
      Date Day  Predicted  Revenue_INR
2019-10-09 Wed        3.4        101.0
2019-10-10 Thu        4.0        119.0
2019-10-11 Fri        4.2        127.0
2019-10-12 Sat        4.2        127.0
2019-10-13 Sun      

In [9]:
# ══════════════════════════════════════════════
#  MONTHLY PROCUREMENT PLAN (Indian Format)
# ══════════════════════════════════════════════

def generate_procurement_plan(df, drug_meta, months_ahead=3):
    """Generate procurement plan for the next N months in Indian pharmacy format."""
    from datetime import date
    today = df['date'].max()
    rows = []

    for code in drug_cols:
        meta   = drug_meta[code]
        ddf    = df[df['drug_code'] == code].sort_values('date')
        avg_30 = ddf['roll_mean_30'].iloc[-1]

        for m in range(1, months_ahead + 1):
            target_month  = (today.month + m - 1) % 12 + 1
            target_year   = today.year + (today.month + m - 1) // 12
            season_mult   = INDIAN_SEASONAL_MULTIPLIER.get(target_month, 1.0)
            drug_peak     = 1.35 if target_month in meta['seasonal_peaks'] else 1.0
            days_in_month = 30

            projected = avg_30 * season_mult * drug_peak * days_in_month
            buffer    = projected * 0.15   # 15% buffer
            order_qty = round(projected + buffer)
            cost_inr  = round(order_qty * meta['mrp_per_strip'] / meta['units_per_strip'] * 0.80)  # ~20% trade margin

            rows.append({
                'Month':        f'{target_year}-{target_month:02d}',
                'Drug':         code,
                'Generic':      meta['generic'],
                'Brand':        meta['top_brand'],
                'Projected Qty':round(projected),
                'Order Qty (+15% buffer)': order_qty,
                'Est. Cost (₹)': cost_inr,
                'Season Mult':  f'{season_mult:.2f}x',
                'Peak Month?':  '✅ YES' if target_month in meta['seasonal_peaks'] else 'No',
                'Lead Time':    f'{meta["lead_time_days"]}d',
                'Schedule':     meta['schedule'],
            })

    return pd.DataFrame(rows)

proc = generate_procurement_plan(df, INDIAN_DRUG_META, months_ahead=3)

print('📋 3-MONTH PROCUREMENT PLAN — Indian Pharmacy')
print('=' * 80)
print(proc.to_string(index=False))

print(f'\n💰 Total Procurement Budget (3 months): ₹{proc["Est. Cost (₹)"].sum():,.0f}')
per_month = proc.groupby('Month')['Est. Cost (₹)'].sum()
print('Monthly breakdown:')
for m, cost in per_month.items():
    print(f'   {m}: ₹{cost:,.0f}')

📋 3-MONTH PROCUREMENT PLAN — Indian Pharmacy
  Month  Drug                                           Generic            Brand  Projected Qty  Order Qty (+15% buffer)  Est. Cost (₹) Season Mult Peak Month? Lead Time Schedule
2019-11 M01AB                                        Diclofenac     Voveran 50mg            195                      224            806       1.15x          No        3d        H
2019-12 M01AB                                        Diclofenac     Voveran 50mg            203                      234            842       1.20x          No        3d        H
2020-01 M01AB                                        Diclofenac     Voveran 50mg            252                      289           1040       1.10x       ✅ YES        3d        H
2019-11 M01AE                                         Ibuprofen     Brufen 400mg            140                      161            258       1.15x          No        3d      OTC
2019-12 M01AE                                         Ibupro

In [10]:
# ══════════════════════════════════════════════
#  REAL-TIME PREDICTION FUNCTION
#  (plug into your HMS backend)
# ══════════════════════════════════════════════

def predict_stock_and_demand(drug_code: str, current_stock: float,
                              date_str: str = None) -> dict:
    """
    Real-time prediction for a drug in your HMS.

    Args:
        drug_code:     one of M01AB, M01AE, N02BA, N02BE, N05B, N05C, R03, R06
        current_stock: actual units on shelf right now
        date_str:      'YYYY-MM-DD' (defaults to today)

    Returns dict with:
        - predicted_demand_tomorrow
        - predicted_demand_7d
        - stock_status
        - days_of_stock
        - reorder_needed (bool)
        - recommended_order_qty
        - top_brand, mrp, schedule
    """
    if drug_code not in INDIAN_DRUG_META:
        return {'error': f'Unknown drug code. Use one of: {list(INDIAN_DRUG_META.keys())}'}

    date  = pd.to_datetime(date_str or pd.Timestamp.today().date())
    meta  = INDIAN_DRUG_META[drug_code]
    ddf   = df[df['drug_code'] == drug_code].sort_values('date')
    last  = ddf.iloc[-1].copy()

    # Prepare input for demand model
    inp = last[FEATURES].copy()
    inp.update({
        'day_of_week':        date.dayofweek,
        'day_of_month':       date.day,
        'month':              date.month,
        'week_of_year':       date.isocalendar()[1],
        'quarter':            (date.month - 1) // 3 + 1,
        'is_weekend':         int(date.dayofweek >= 5),
        'is_monsoon':         int(date.month in [6, 7, 8, 9]),
        'indian_season_mult': INDIAN_SEASONAL_MULTIPLIER.get(date.month, 1.0),
        'stock_level':        current_stock,
    })

    demand_tomorrow = max(0, model_demand.predict(pd.DataFrame([inp]))[0])
    avg_demand      = last['roll_mean_30']
    demand_7d       = demand_tomorrow * 7 * INDIAN_SEASONAL_MULTIPLIER.get(date.month, 1.0)
    dos             = current_stock / (avg_demand + 0.01)
    rop             = last['reorder_point']
    ss              = last['safety_stock']
    eoq             = np.sqrt((2 * avg_demand * 365 * 500) / (meta['mrp_per_strip'] * 0.25 / 365 + 0.01))

    if dos < 3:   status = 'CRITICAL — ORDER IMMEDIATELY'
    elif dos < 7: status = 'LOW — Order within 24 hours'
    elif dos > 45: status = 'OVERSTOCK — Reduce next order'
    else:         status = 'OK'

    return {
        'drug_code':              drug_code,
        'generic_name':           meta['generic'],
        'top_brand':              meta['top_brand'],
        'schedule':               meta['schedule'],
        'mrp_per_strip_inr':      meta['mrp_per_strip'],
        'current_stock':          round(current_stock),
        'days_of_stock':          round(dos, 1),
        'predicted_demand_tomorrow': round(demand_tomorrow, 1),
        'predicted_demand_7d':    round(demand_7d, 1),
        'avg_demand_30d':         round(avg_demand, 1),
        'stock_status':           status,
        'reorder_point':          round(rop),
        'safety_stock':           round(ss),
        'recommended_order_qty':  round(max(eoq, avg_demand * 14)),
        'est_order_cost_inr':     round(max(eoq, avg_demand*14) * meta['mrp_per_strip'] / meta['units_per_strip'] * 0.80),
        'disease_season':         meta['disease_season'],
    }

# ── Demo ──
print('🔍 REAL-TIME PREDICTION DEMO')
print('=' * 55)
for code in ['N02BE', 'R03', 'M01AB']:
    result = predict_stock_and_demand(code, current_stock=150)
    print(f"\n{result['drug_code']} — {result['generic_name']} ({result['top_brand']})")
    print(f"  Stock       : {result['current_stock']} units ({result['days_of_stock']} days)")
    print(f"  Status      : {result['stock_status']}")
    print(f"  Tomorrow    : {result['predicted_demand_tomorrow']} units")
    print(f"  Next 7 days : {result['predicted_demand_7d']} units")
    print(f"  Order now   : {result['recommended_order_qty']} units (₹{result['est_order_cost_inr']:,})")
    print(f"  Season note : {result['disease_season']}")

🔍 REAL-TIME PREDICTION DEMO

N02BE — Paracetamol (Dolo 650)
  Stock       : 150 units (4.1 days)
  Status      : LOW — Order within 24 hours
  Tomorrow    : 41.099998474121094 units
  Next 7 days : 302.2 units
  Order now   : 20832 units (₹33,331)
  Season note : Peak during monsoon (dengue/malaria fever), also winter flu

R03 — Bronchodilators (Salbutamol / Budesonide) (Asthalin Inhaler)
  Stock       : 150 units (32.2 days)
  Status      : OK
  Tomorrow    : 6.5 units
  Next 7 days : 47.5 units
  Order now   : 3567 units (₹513,701)
  Season note : Winter smog (Delhi/NCR), monsoon mold spores

M01AB — Diclofenac (Voveran 50mg)
  Stock       : 150 units (26.5 days)
  Status      : OK
  Tomorrow    : 5.0 units
  Next 7 days : 36.8 units
  Order now   : 7106 units (₹25,581)
  Season note : Arthritis flares in winter, muscle pain year-round


In [11]:
# ══════════════════════════════════════════════
#  SAVE MODELS FOR HMS INTEGRATION
# ══════════════════════════════════════════════

Path('models').mkdir(exist_ok=True)
pickle.dump(model_demand,  open('models/indian_pharma_demand_model.pkl', 'wb'))
pickle.dump(model_clf,     open('models/indian_pharma_stock_classifier.pkl', 'wb'))
pickle.dump(le_status,     open('models/stock_label_encoder.pkl', 'wb'))
pickle.dump(le_drug,       open('models/drug_label_encoder.pkl', 'wb'))
pickle.dump(FEATURES,      open('models/feature_list.pkl', 'wb'))

# Save Indian drug metadata as JSON for HMS frontend
with open('models/indian_drug_meta.json', 'w') as f:
    json.dump(INDIAN_DRUG_META, f, indent=2)

print('✅ All models saved to models/')
print('   indian_pharma_demand_model.pkl   — XGBoost demand forecaster')
print('   indian_pharma_stock_classifier.pkl — Stock status classifier')
print('   indian_drug_meta.json           — Indian brand/price metadata')
print('\nLoad in your HMS backend:')
print('   import pickle')
print('   model = pickle.load(open("models/indian_pharma_demand_model.pkl", "rb"))')

✅ All models saved to models/
   indian_pharma_demand_model.pkl   — XGBoost demand forecaster
   indian_pharma_stock_classifier.pkl — Stock status classifier
   indian_drug_meta.json           — Indian brand/price metadata

Load in your HMS backend:
   import pickle
   model = pickle.load(open("models/indian_pharma_demand_model.pkl", "rb"))
